# Batch Processing with Config File

This notebook demonstrates how to use the server's batch processing functionality with custom configuration files.

## Prerequisites

1. Server must be running: `python run_server.py`
2. Environment variables must be set (NEO4J credentials, API keys, etc.)
3. Input data files must be available

## 1. Setup and Imports

In [1]:
import yaml
import requests
import os
import json
from pathlib import Path
from pprint import pprint
from datetime import datetime

## 2. Configuration

In [2]:
# Server configuration
SERVER_URL = "http://localhost:8000"

# Paths
CONFIG_PATH = "../configs/financebench_pipeline.yaml"
INPUT_PATH = "../data/financebench"
OUTPUT_PATH = "../output/batch_results"

# Batch processing parameters
BATCH_SIZE = 10  # Number of files to process per batch

print(f"✅ Configuration set:")
print(f"   Server: {SERVER_URL}")
print(f"   Config: {CONFIG_PATH}")
print(f"   Input: {INPUT_PATH}")
print(f"   Output: {OUTPUT_PATH}")
print(f"   Batch size: {BATCH_SIZE}")

✅ Configuration set:
   Server: http://localhost:8000
   Config: ../configs/financebench_pipeline.yaml
   Input: ../data/financebench
   Output: ../output/batch_results
   Batch size: 10


## 3. Check Server Status

In [15]:
try:
    response = requests.get(f"{SERVER_URL}/health", timeout=5)
    if response.status_code == 200:
        print("✅ Server is running!")
        print(f"   Response: {response.json()}")
    else:
        print(f"⚠️  Server returned status: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("❌ Server is not running!")
    print("   Please start the server with: python run_server.py")
except Exception as e:
    print(f"❌ Error checking server: {e}")

✅ Server is running!
   Response: {'status': 'healthy', 'timestamp': '2025-10-19T02:08:22.084135', 'service': 'kag-langgraph-server'}


## 4. Helper Functions

In [4]:
def load_yaml_config(config_path: str):
    """
    Load YAML config and resolve environment variables.
    """
    print(f"📄 Loading config from: {config_path}")
    
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # Resolve environment variables
    config = resolve_env_vars(config)
    
    print(f"✅ Config loaded successfully")
    return config


def resolve_env_vars(obj):
    """
    Recursively resolve ${VAR_NAME} in config.
    """
    if isinstance(obj, dict):
        return {k: resolve_env_vars(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [resolve_env_vars(item) for item in obj]
    elif isinstance(obj, str) and obj.startswith('${') and obj.endswith('}'):
        # Handle both ${VAR} and ${VAR:default}
        var_part = obj[2:-1]
        if ':' in var_part:
            var_name, default = var_part.split(':', 1)
            return os.getenv(var_name, default)
        else:
            var_name = var_part
            return os.getenv(var_name, obj)
    return obj


def format_metrics(metrics: dict):
    """
    Format metrics for display.
    """
    if not metrics:
        return "No metrics available"
    
    output = []
    output.append("📈 Metrics:")
    output.append(f"   Files processed: {metrics.get('files_processed', 0)}")
    output.append(f"   Total entities: {metrics.get('total_entities', 0)}")
    output.append(f"   Total relationships: {metrics.get('total_relationships', 0)}")
    output.append(f"   Processing time: {metrics.get('processing_time', 0):.2f}s")
    return "\n".join(output)


print("✅ Helper functions defined")

✅ Helper functions defined


## 5. Load Configuration File

In [19]:
# Load the YAML config
config = load_yaml_config(CONFIG_PATH)

# Display config structure (hide sensitive info)
print("\n📋 Config Structure:")
print(f"   Pipeline name: {config.get('pipeline', {}).get('name', 'N/A')}")
print(f"   Components: {list(config.get('pipeline', {}).get('components', {}).keys())}")

# Show enabled components
print("\n🔧 Enabled Components:")
for name, component in config.get('pipeline', {}).get('components', {}).items():
    enabled = component.get('enabled', False)
    comp_type = component.get('type', 'unknown')
    status = "✅" if enabled else "❌"
    print(f"   {status} {name}: {comp_type}")

📄 Loading config from: ../configs/financebench_pipeline.yaml
✅ Config loaded successfully

📋 Config Structure:
   Pipeline name: financebench_extraction_pipeline
   Components: ['scanner', 'reader', 'splitter', 'extractor', 'vectorizer', 'writer']

🔧 Enabled Components:
   ✅ scanner: file_scanner
   ✅ reader: financebench_reader
   ❌ splitter: semantic_splitter
   ✅ extractor: llm_extractor
   ✅ vectorizer: gemini_vectorizer
   ✅ writer: neo4j_writer


## 6. Check Input Files

In [17]:
input_dir = Path(INPUT_PATH)

if input_dir.exists():
    # Count files
    files = list(input_dir.glob('*.txt'))
    print(f"✅ Input directory exists: {input_dir}")
    print(f"   Total .txt files: {len(files)}")
    
    if len(files) > 0:
        print(f"\n📄 Sample files (first 5):")
        for i, file in enumerate(files[:5], 1):
            size_kb = file.stat().st_size / 1024
            print(f"   {i}. {file.name} ({size_kb:.1f} KB)")
    else:
        print("   ⚠️  No .txt files found!")
else:
    print(f"❌ Input directory does not exist: {input_dir}")
    print("   Please create the directory and add files")

✅ Input directory exists: ../data/financebench
   Total .txt files: 150

📄 Sample files (first 5):
   1. financebench_id_02987.txt (7.2 KB)
   2. financebench_id_00606.txt (4.8 KB)
   3. financebench_id_00438.txt (2.6 KB)
   4. financebench_id_00216.txt (4.0 KB)
   5. financebench_id_00941.txt (3.5 KB)


## 7. Run Batch Processing

In [ ]:
# Prepare the request
request_data = {
    'input_path': str(Path(INPUT_PATH).absolute()),
    'config': config,
    'batch_size': BATCH_SIZE,
    'output_path': str(Path(OUTPUT_PATH).absolute())
}

print("🚀 Starting batch processing...")
print(f"   Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Endpoint: {SERVER_URL}/pipeline/run-batch")
print(f"   Batch size: {BATCH_SIZE} files per batch")
print("\n⏳ Please wait... (this may take several minutes)")

try:
    response = requests.post(
        f'{SERVER_URL}/pipeline/run-batch',
        json=request_data,
        # timeout=3600  # 1 hour timeout
    )
    
    print(f"\n📡 Response received (Status: {response.status_code})")
    
    if response.status_code == 200:
        result = response.json()
        print("\n" + "="*70)
        print("✅ BATCH PROCESSING COMPLETED!")
        print("="*70)
    else:
        print("\n" + "="*70)
        print(f"❌ BATCH PROCESSING FAILED (Status: {response.status_code})")
        print("="*70)
        print(f"\nError: {response.text}")
        result = None
        
except requests.exceptions.Timeout:
    print("\n❌ Request timed out after 1 hour")
    result = None
except Exception as e:
    print(f"\n❌ Error: {e}")
    result = None

🚀 Starting batch processing...
   Time: 2025-10-19 09:13:48
   Endpoint: http://localhost:8000/pipeline/run-batch
   Batch size: 10 files per batch

⏳ Please wait... (this may take several minutes)

❌ Request timed out after 1 hour


## 8. Display Results

In [22]:
if result:
    print("\n📊 RESULTS SUMMARY")
    print("="*70)
    
    # Basic info
    print(f"\n🆔 Pipeline ID: {result.get('pipeline_id', 'N/A')}")
    print(f"📈 Status: {result.get('status', 'unknown')}")
    print(f"⏱️  Timestamp: {result.get('timestamp', 'N/A')}")
    
    # Metrics
    metrics = result.get('metrics', {})
    if metrics:
        print(f"\n{format_metrics(metrics)}")
    
    # Execution summary
    exec_summary = result.get('execution_summary', {})
    if exec_summary:
        print("\n🔄 Execution Summary:")
        print(f"   Total batches: {exec_summary.get('batches', 0)}")
        print(f"   Files per batch: {exec_summary.get('files_per_batch', 0)}")
        print(f"   Successful files: {exec_summary.get('successful_files', 0)}")
        print(f"   Failed files: {exec_summary.get('failed_files', 0)}")
    
    # Errors
    errors = result.get('errors', [])
    if errors:
        print(f"\n⚠️  Errors ({len(errors)}):")
        for i, error in enumerate(errors[:5], 1):
            print(f"   {i}. {error}")
        if len(errors) > 5:
            print(f"   ... and {len(errors) - 5} more errors")
    else:
        print("\n✅ No errors!")
    
    # Output path
    print(f"\n📁 Output Location:")
    print(f"   {result.get('output_path', 'N/A')}")
    
    print("\n" + "="*70)
else:
    print("\n❌ No results to display (processing failed or was cancelled)")


❌ No results to display (processing failed or was cancelled)


## 9. Verify Output Files

In [23]:
if result and result.get('output_path'):
    output_dir = Path(result['output_path'])
    
    if output_dir.exists():
        # List output files
        output_files = list(output_dir.glob('**/*'))
        output_files = [f for f in output_files if f.is_file()]
        
        print(f"📂 Output Directory: {output_dir}")
        print(f"   Total files created: {len(output_files)}")
        
        if output_files:
            print("\n📄 Output Files (first 10):")
            for i, file in enumerate(output_files[:10], 1):
                size_kb = file.stat().st_size / 1024
                print(f"   {i}. {file.name} ({size_kb:.1f} KB)")
        else:
            print("   ⚠️  No output files found")
    else:
        print(f"⚠️  Output directory does not exist: {output_dir}")
else:
    print("ℹ️  No output path available")

ℹ️  No output path available


## 10. Advanced: Custom Config Modifications

You can modify the config programmatically before sending the request:

In [ ]:
# Example: Modify config for testing (small batch)
test_config = load_yaml_config(CONFIG_PATH)

# Limit to only 3 files for testing
test_config['pipeline']['components']['scanner']['config']['max_files'] = 3

# Use a different database
# test_config['pipeline']['components']['writer']['config']['database'] = 'test_db'

print("🔧 Modified Config:")
print(f"   Max files: {test_config['pipeline']['components']['scanner']['config']['max_files']}")
print(f"   Database: {test_config['pipeline']['components']['writer']['config']['database']}")

# Uncomment to run with test config
# response = requests.post(
#     f'{SERVER_URL}/pipeline/run-batch',
#     json={
#         'input_path': str(Path(INPUT_PATH).absolute()),
#         'config': test_config,
#         'batch_size': 2
#     },
#     timeout=1800
# )

## 11. Check Neo4j Database

Verify that data was written to Neo4j:

In [ ]:
# Optional: Check Neo4j if you have neo4j driver installed
try:
    from neo4j import GraphDatabase
    
    NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
    NEO4J_USERNAME = os.getenv('NEO4J_USERNAME', 'neo4j')
    NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
    NEO4J_DATABASE = config['pipeline']['components']['writer']['config']['database']
    
    if NEO4J_PASSWORD:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
        
        with driver.session(database=NEO4J_DATABASE) as session:
            # Count entities
            result_entities = session.run("MATCH (n) RETURN count(n) as count")
            entity_count = result_entities.single()['count']
            
            # Count relationships
            result_rels = session.run("MATCH ()-[r]->() RETURN count(r) as count")
            rel_count = result_rels.single()['count']
            
            print("🗄️  Neo4j Database:")
            print(f"   Database: {NEO4J_DATABASE}")
            print(f"   Total nodes: {entity_count}")
            print(f"   Total relationships: {rel_count}")
        
        driver.close()
    else:
        print("⚠️  NEO4J_PASSWORD not set, skipping database check")
        
except ImportError:
    print("ℹ️  neo4j driver not installed, skipping database check")
    print("   Install with: pip install neo4j")
except Exception as e:
    print(f"⚠️  Could not connect to Neo4j: {e}")

## 12. Summary and Next Steps

In [ ]:
print("\n" + "="*70)
print("📝 SUMMARY")
print("="*70)
print("\n✅ Completed Steps:")
print("   1. Loaded configuration from YAML file")
print("   2. Checked server status")
print("   3. Verified input files")
print("   4. Ran batch processing")
print("   5. Verified results")

if result and result.get('status') == 'completed':
    print("\n🎉 Batch processing completed successfully!")
    print("\n💡 Next Steps:")
    print("   - Check the output files in:", result.get('output_path'))
    print("   - Query Neo4j database to explore the knowledge graph")
    print("   - Use the RAG chatbot to query the data")
    print("   - Run langgraph dev to interact with the graph")
else:
    print("\n⚠️  Batch processing did not complete successfully")
    print("\n💡 Troubleshooting:")
    print("   - Check server logs for errors")
    print("   - Verify environment variables are set correctly")
    print("   - Check Neo4j connection")
    print("   - Review the config file for errors")

print("\n" + "="*70)